In [1]:
#Project on linux systems

In [2]:
import sys
try:
    import kagglehub
    import warnings
    import os
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import torch
    import torch.nn as nn
    import torchvision.transforms.v2 as T
    from IPython.display import clear_output
    from PIL import Image
    from torch.utils.data import DataLoader
    from tqdm import tqdm
    from torch.optim import Optimizer
    from torchmetrics.classification import BinaryF1Score
    import os
    from sklearn.model_selection import train_test_split
    from torch.utils.data import Dataset, DataLoader
    import torch.nn.functional as F
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    from torch.nn.modules.linear import Linear
    from torch.nn.modules.flatten import Flatten
    from torch.nn.modules.activation import GELU
    from torch.nn.modules.batchnorm import BatchNorm2d
    from torch.optim.lr_scheduler import OneCycleLR
    from sklearn.metrics import f1_score
    import cv2
    import numpy as np
except ImportError as e:
    print(f"Error {e} ")
    !pip install kagglehub torch torchvision numpy pandas matplotlib scikit-learn tqdm pillow torchmetrics kagglehub

In [3]:
##GLOBAL_OPTIONS
LOGMODE=False
INPUTPATH="ConvMixerOneCyclerNormalize.pt"#ссылка на предобученную модель для подбора threshold
FINDTHRESHOLD=True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")#посмотри и улыбнись
src_path = "/home/alex/.cache/kagglehub/competitions/ml-intensive-yandex-academy-spring-2026"# ссылка до директории с датасетом
KAGGLEHUB=False# Если вычисляешь на kaggle
OUTPUTPATH="./"#куда будет записываться модельки
NUMEPOCHS=100
DOFIT=True
device
DATASETDOWNLOAD=False# скачан ли датасет на компьютер
if not DATASETDOWNLOAD:
  kagglehub.login()


###

In [4]:
src_path = kagglehub.competition_download('ml-intensive-yandex-academy-spring-2026')

In [5]:
def train(model: nn.Module, data_loader: DataLoader, optimizer: Optimizer, loss_fn, device: torch.device,scheduler):
    model.train()
    total_loss = 0.0
    for x, y in tqdm(data_loader) if not LOGMODE else data_loader:
        x = x.to(device)
        y = y.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)

@torch.inference_mode()
def evaluate(model: nn.Module, data_loader: DataLoader, loss_fn, device: torch.device):
    model.eval()
    total_loss = 0.0
    f1_metric = BinaryF1Score().to(device)
    for x, y in tqdm(data_loader) if not LOGMODE else data_loader:

        x = x.to(device)
        y = y.to(device)

        output = model(x)

        y_for_loss = y.float().unsqueeze(1)
        loss = loss_fn(output, y_for_loss)
        total_loss += loss.item()


        probs = torch.sigmoid(output)


        y_for_metric = y.unsqueeze(1)


        f1_metric.update(probs, y_for_metric)

    return total_loss / len(data_loader), f1_metric.compute().item()

def fit(model, train_loader, valid_loader, optimizer,scheduler, loss_fn, device, num_epochs, title):
    train_loss_history, valid_loss_history = [], []
    valid_f1_history =[]

    best_valid_f1 = -1.0

    for epoch in range(num_epochs):
        train_loss = train(model, train_loader, optimizer, loss_fn, device,scheduler)
        valid_loss, valid_f1 = evaluate(model, valid_loader, loss_fn, device)
        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        valid_f1_history.append(valid_f1)

        clear_output()

        plot_stats(
            train_loss_history, valid_loss_history,
            valid_f1_history,
            title
        )
        if valid_f1 > best_valid_f1:
            best_valid_f1 = valid_f1
            torch.save(model.state_dict(), "checkpoints/best_model.pt")
            torch.save(optimizer.state_dict(), "checkpoints/best_optimizer.pt")
            print(f"best model epoch {epoch}")
    return train_loss_history, valid_loss_history,valid_f1_history

In [6]:
def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    valid_f1: list[float],
    title: str
):
    plt.figure(figsize=(16, 8))

    plt.title(title + ' loss')

    plt.plot(train_loss, label='Train loss')
    plt.plot(valid_loss, label='Valid loss')
    plt.legend()

    plt.show()

    plt.figure(figsize=(16, 8))

    plt.title(title + ' f1')

    plt.plot(valid_f1, label='Valid f1')
    plt.legend()

    plt.show()
    print("1: train loss 2: valid loss 3: valid f1")
    print(train_loss,valid_loss,valid_f1,sep='\n', flush=True)

In [7]:
@torch.inference_mode()
def predict_test(model, test_loader, device, threshold=0.5):
    model.eval()

    all_ids = []
    all_preds = []

    for x, ids in tqdm(test_loader):
        x = x.to(device)

        output = model(x)
        res = torch.sigmoid(output).squeeze(1)
        preds = (res > threshold).long()

        all_ids.extend(ids.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

    submission = pd.DataFrame({
        "id": all_ids,
        "target_feature": all_preds
    })

    submission = submission.sort_values("id").reset_index(drop=True)# сортирует по возрастанию id удаляет старые индексы и созает новые
    submission.to_csv(f"{OUTPUTPATH}submission.csv", index=False)

    return submission

In [8]:
def add_laplacian(image):# Делает изображение чб, нормализует
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_32F)

    lap = (lap - lap.mean()) / (lap.std() + 1e-6)

    lap = np.expand_dims(lap, axis=2)
    image = image.astype(np.float32)

    return np.concatenate([image, lap], axis=2)

def make_test_image_loader(
    folder_path,
    transform=None,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    return_ids=True,
):
    image_files = sorted(
        [f for f in os.listdir(folder_path) if f.lower().endswith(".jpg")],
        key=lambda x: int(os.path.splitext(x)[0])
    )

    data = []

    for filename in image_files:
        path = os.path.join(folder_path, filename)
        image = np.array(Image.open(path).convert("RGB"))
        image = add_laplacian(image)
        if transform is not None:
            image = transform(image=image)["image"]

        if return_ids:
            img_id = int(os.path.splitext(filename)[0])
            data.append((image, img_id))
        else:
            data.append(image)

    return DataLoader(
        data,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True
    )

def make_train_valid_loaders(
    images_folder,
    labels_csv,
    train_transform=None,
    valid_transform=None,
    batch_size=32,
    valid_size=0.2,
    random_state=42,
    shuffle_train=True,
    num_workers=2
):
    df = pd.read_csv(labels_csv, header=None, names=["Id", "target_feature"])

    train_df, valid_df = train_test_split(
        df,
        test_size=valid_size,
        random_state=random_state,
        stratify=df["target_feature"]
    )

    train_items = [
        (os.path.join(images_folder, f"{int(row.Id)}.jpg"), int(row.target_feature))
        for row in train_df.itertuples(index=False)
    ]
    valid_items = [
        (os.path.join(images_folder, f"{int(row.Id)}.jpg"), int(row.target_feature))
        for row in valid_df.itertuples(index=False)
    ]

    def make_collate(transform):
        def collate_fn(batch):
            images = []
            labels = []

            for path, label in batch:
                image = np.array(Image.open(path).convert("RGB"))
                if transform is not None:
                    image = transform(image=image)["image"]

                image = image.permute(1, 2, 0).cpu().numpy()

                image = add_laplacian(image)

                image = torch.from_numpy(image).permute(2, 0, 1).float()
                images.append(image)
                labels.append(label)

            images = torch.stack(images)
            labels = torch.tensor(labels, dtype=torch.long)
            return images, labels
        return collate_fn

    train_loader = DataLoader(
        train_items,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=num_workers,
        collate_fn=make_collate(train_transform),
        pin_memory=True
    )

    valid_loader = DataLoader(
        valid_items,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=make_collate(valid_transform),
        pin_memory=True
    )

    return train_loader, valid_loader

In [9]:
class RawImageDataset(Dataset):
    def __init__(self, images_folder, labels_csv):
        df = pd.read_csv(labels_csv, header=None, names=["Id", "target_feature"])
        self.image_paths = [os.path.join(images_folder, f"{int(row.Id)}.jpg") for row in df.itertuples(index=False)]
        self.transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)]) # scale=True делит на 255

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        return self.transform(image)

def CalcMeanStd():
    calc_dataset = RawImageDataset(
        images_folder=f"{src_path}/dataset/train_images",
        labels_csv=f"{src_path}/dataset/train_solution.csv"
    )
    calc_loader = DataLoader(calc_dataset, batch_size=64, shuffle=False, num_workers=2)
    mean = 0.
    std = 0.
    n_samples = 0.

    for images in tqdm(calc_loader):
        batch_samples = images.size(0)
        images = images.view(batch_samples, images.size(1), -1)
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        n_samples += batch_samples

    mean /= n_samples
    std /= n_samples

    print(f"Mean: {mean}")
    print(f"Std: {std}")
    return mean,std


In [10]:
@torch.inference_mode()
def predict_with_tta(model,original_loader,tta_loader,device,threshold=0.5):
    model.eval()
    all_ids_list = []
    original_probs_list = []
    tta_probs_list = []

    for x, ids in tqdm(original_loader):
        x = x.to(device)
        output = model(x)
        original_probs_list.append(torch.sigmoid(output).cpu())
        all_ids_list.append(ids.cpu())

    for x, _ in tqdm(tta_loader):
        x = x.to(device)
        output = model(x)
        tta_probs_list.append(torch.sigmoid(output).cpu())

    all_ids = torch.cat(all_ids_list).numpy()
    original_probs = torch.cat(original_probs_list).numpy().flatten()
    tta_probs = torch.cat(tta_probs_list).numpy().flatten()
    final_probs = (original_probs + tta_probs) / 2


    preds=(final_probs>threshold).astype(int)
    submission = pd.DataFrame({"id": all_ids, "target_feature": preds})
    submission = submission.sort_values("id").reset_index(drop=True)
    submission.to_csv("submission_tta.csv", index=False)
    return submission

In [11]:
class Baseline(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class BlockConvMixer(nn.Module):
  def __init__(self,dim,kernel_size=5):
    super().__init__()
    self.depthwise=nn.Sequential(
        nn.Conv2d(dim,dim,kernel_size,padding=kernel_size//2,groups=dim),#отличие convmixer от Cnn
        nn.GELU(),# не обрезат отриц часть
        nn.BatchNorm2d(dim)
    )
    self.pointwise=nn.Sequential(
        nn.Conv2d(dim,dim,kernel_size=1),# на этом этапе обьединяем слои по 1
        nn.GELU(),
        nn.BatchNorm2d(dim)
    )
  def forward(self,x):
    x=x+self.depthwise(x)
    x=self.pointwise(x)
    return x


class ConvMixer(nn.Module):
  def __init__(self,dim=384,depth=12,kernel_size=5,patch_size=8,n_classes=1):
    super().__init__()
    self.patchEmb=nn.Sequential(
        nn.Conv2d(4,dim,kernel_size=patch_size,stride=patch_size),
        nn.GELU(),
        nn.BatchNorm2d(dim)
    )
    self.blocks=nn.Sequential(
        *[BlockConvMixer(dim,kernel_size) for _ in range(depth)]
    )
    self.head=nn.Sequential(
        nn.AdaptiveAvgPool2d((1,1)),
        nn.Dropout(0.6),
        nn.Flatten(),

        nn.Linear(dim,n_classes)
    )
    self.norm = nn.BatchNorm2d(4)
  def forward(self,x):
    x = self.norm(x)
    x=self.patchEmb(x)
    x=self.blocks(x)
    x=self.head(x)
    return x

In [12]:

##TTA проверяем тест и на повернутых изображениях т.к модель обучена на повернутых
tta_transforms=A.Compose([
    A.HorizontalFlip(p=1.0),
    A.VerticalFlip(p=1.0),
    A.RandomRotate90(p=1.0),
    A.MedianBlur(blur_limit=3, p=0.3),
    ToTensorV2()
])
tta_test_loader=make_test_image_loader(
    f"{src_path}/dataset/test_images",
    transform=tta_transforms,
    batch_size=64,
    shuffle=False
    )

train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),

    A.ShiftScaleRotate(
        shift_limit=0.03,
        scale_limit=0.05,
        rotate_limit=10,
        p=0.4
    ),

    A.OneOf([
        A.GaussNoise(var_limit=(5.0, 30.0), p=1.0),
        A.ISONoise(p=1.0),
    ], p=0.5),

    A.ImageCompression(quality_lower=60, quality_upper=100, p=0.5),

    A.OneOf([
        A.RandomBrightnessContrast(p=1.0),
        A.HueSaturationValue(p=1.0),
    ], p=0.3),
    A.MedianBlur(blur_limit=3, p=0.3),
    ToTensorV2(),
])
test_transforms = A.Compose([
    A.MedianBlur(blur_limit=3, p=0.3),
    ToTensorV2(),
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_6449/1822358098.py:27: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 30.0), p=1.0),
/tmp/ipykernel_6449/1822358098.py:31: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=60, quality_upper=100, p=0.5),


In [ ]:
mean,std=CalcMeanStd()
train_loader, valid_loader = make_train_valid_loaders(images_folder=f"{src_path}/dataset/train_images",labels_csv=f"{src_path}/dataset/train_solution.csv",train_transform=train_transforms,valid_transform=test_transforms,batch_size=16,valid_size=0.2,random_state=100,shuffle_train=True, num_workers=2)
test_loader = make_test_image_loader(f"{src_path}/dataset/test_images",transform=test_transforms, batch_size=16, shuffle= False,  num_workers=2)


100%|██████████| 782/782 [02:21<00:00,  5.51it/s]


Mean: tensor([0.5192, 0.4276, 0.3844])
Std: tensor([0.2587, 0.2372, 0.2334])


In [ ]:
img=Image.open(str(train_loader.dataset[0][0]))
display(img)
img.size


In [ ]:
df = pd.read_csv(f"{src_path}/dataset/train_solution.csv", header=None, names=["Id", "target_feature"])

num_pos = (df["target_feature"] == 1).sum()
num_neg = (df["target_feature"] == 0).sum()
print(num_pos,num_neg)

pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)



#model = Baseline().to(device)
model=ConvMixer().to(device)

df = pd.read_csv(f"{src_path}/dataset/train_solution.csv", header=None, names=["Id", "target_feature"])
num_total = len(df)
num_pos = (df["target_feature"] == 1).sum()

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-2)
scheduler = OneCycleLR(
    optimizer,
    max_lr=5e-4,
    steps_per_epoch=len(train_loader),
    epochs=NUMEPOCHS,
    pct_start=0.3,
)
os.makedirs("checkpoints", exist_ok=True)

In [ ]:
data=fit(model, train_loader, valid_loader, optimizer,scheduler, loss_fn, device, num_epochs=NUMEPOCHS, title="baseline") if DOFIT==True else None


In [ ]:
torch.save(model.state_dict(), f"{OUTPUTPATH}/ConvMixerOneCyclerNormalize.pt")

if LOGMODE==True and FINDTHRESHOLD==False:
    with open(f'{OUTPUTPATH}/score.txt', 'w', encoding='utf-8') as f:
        f.write(f'1:train loss 2: test loss 3: f1\n')
        f.write(f'{data[0]}\n')
        f.write(f'{data[1]}\n')
        f.write(f'{data[2]}\n')

In [ ]:
# подбиарем threshold
if FINDTHRESHOLD==True:
    model=ConvMixer().to(device)
    state_dict = torch.load(INPUTPATH, map_location=device)

    old_weight = state_dict["patchEmb.0.weight"]

    if old_weight.shape[1] == 3:
        new_weight = torch.zeros((old_weight.shape[0], 4, 8, 8))
        new_weight[:, :3, :, :] = old_weight
        new_weight[:, 3:4, :, :] = old_weight.mean(dim=1, keepdim=True)
        state_dict["patchEmb.0.weight"] = new_weight

    model.load_state_dict(state_dict)
    #model.load_state_dict(torch.load(INPUTPATH, map_location=device))
    model.to(device)
    model.eval()
    print("Model is ready for finding best threshold")

    all_probs = []
    all_targets = []


    with torch.no_grad():
        for x,y in tqdm(valid_loader) if not LOGMODE else valid_loader:
            x=x.to(device)
            output=model(x)
            probs = torch.sigmoid(output).cpu().numpy()

            all_probs.extend(probs)
            all_targets.extend(y.cpu().numpy())
    all_probs = np.array(all_probs).flatten()
    all_targets = np.array(all_targets)
    best_f1 = 0
    best_threshold = 0
    for threshold in np.arange(0.01 ,1.0, 0.001):
        preds=(all_probs>threshold).astype(int)
        current_f1=f1_score(all_targets,preds)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_threshold = threshold
    print(f"f1 {best_f1}:.6f")
    print(f"threshold {best_threshold}")

In [ ]:
submission=predict_with_tta(model, test_loader,tta_test_loader, device,threshold=best_threshold)
submission.head(10)